# Tabular SARSA: learn from the action you actually take

SARSA estimates the action value of its current behavior policy:

$$
Q^\pi(s,a)=\mathbb{E}_\pi\!\left[\sum_{k=0}^{\infty}\gamma^k r_{t+k+1}\,\middle|\,s_t=s,\,a_t=a\right].
$$

Its name comes from the five values used by an update: **State, Action, Reward, next State, next Action**. Because Taxi has finite discrete spaces, the estimates fit in a table $Q\in\mathbb{R}^{|\mathcal S|\times|\mathcal A|}$.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

TOTAL_TIMESTEPS = 100_000
LEARNING_RATE = 0.2
GAMMA = 0.95
EPSILON_START = 1.0
EPSILON_END = 0.05
EXPLORATION_STEPS = 80_000

env = gym.make("Taxi-v4")
q_table = np.zeros((env.observation_space.n, env.action_space.n), dtype=np.float32)

print("Q-table shape:", q_table.shape)

## 1. Define the behavior policy

SARSA trains with an $\varepsilon$-greedy policy. With probability $\varepsilon_t$ it samples uniformly; otherwise it selects $\arg\max_a Q(s_t,a)$. The exploration rate decays linearly from `EPSILON_START` to `EPSILON_END`.

In [ ]:
def epsilon_at(step):
    fraction = min(step / EXPLORATION_STEPS, 1.0)
    return EPSILON_START + fraction * (EPSILON_END - EPSILON_START)

def greedy_action(state):
    return int(np.argmax(q_table[state]))

def select_action(state, step):
    """Sample one action from the epsilon-greedy behavior policy."""
    epsilon = epsilon_at(step)
    if np.random.random() < epsilon:
        return np.random.choice(q_table.shape[1])
    else:
        return greedy_action(state)

## 2. Write the SARSA update

After observing $(s_t,a_t,r_{t+1},s_{t+1})$, sample $a_{t+1}$ from the same behavior policy and form

$$
\delta_t=r_{t+1}+\gamma(1-d_t)Q(s_{t+1},a_{t+1})-Q(s_t,a_t),
$$

then update

$$
Q(s_t,a_t)\leftarrow Q(s_t,a_t)+\alpha\delta_t.
$$

Here $d_t=1$ only for a true Gymnasium `terminated` transition. A time-limit `truncated` transition still bootstraps from its final observation.

SARSA is **on-policy**: the learning target utilizes the actual next action, instead of estimating it like in Q-Learning.

In [ ]:
def update_q_table(state, action, reward, next_state, next_action, terminated):
    """Apply one SARSA update and return the TD error."""
    y = reward + (1 - terminated) * GAMMA * q_table[next_state, next_action]
    td_error = y - q_table[state, action]
    q_table[state, action] += LEARNING_RATE * td_error

## 3. Preserve the on-policy action chain

One-step SARSA follows this sequence:

1. sample $a_t$ and take it;
2. observe $r_{t+1}$ and $s_{t+1}$;
3. if the MDP did not terminate, sample $a_{t+1}$;
4. update with $(s_t,a_t,r_{t+1},s_{t+1},a_{t+1})$;
5. carry that exact $a_{t+1}$ into the next interaction.

For `truncated=True`, sample a next action for bootstrapping, but reset afterward because the Gymnasium episode ended. For `terminated=True`, there is no next action and no bootstrap value.

> **TODO:** fill in the marked action-selection and action-handoff lines. Sampling a fresh action at the start of every loop would break the SARSA relationship.

In [ ]:
def train(total_timesteps):
    episode_returns = []
    episode_return = 0.0
    state, _ = env.reset()

    action = None
    for step in range(total_timesteps):
        
        if action is None:
            action = select_action(state, step)

        next_state, reward, terminated, truncated, info = env.step(action)
        episode_return += reward

        next_action = select_action(next_state, step + 1)
        
        update_q_table(state, action, reward, next_state, next_action, terminated)

        if terminated or truncated:
            episode_returns.append(episode_return)
            episode_return = 0.0
            state, _ = env.reset()
            action = None
        else:
            state = next_state
            action = None

    env.close()
    return episode_returns

episode_returns = train(TOTAL_TIMESTEPS)
print(f"Trained for {len(episode_returns)} episodes.")

## 4. Check your understanding

Before running training, explain these differences in your own words:

- Why does SARSA use $Q(s_{t+1},a_{t+1})$ while Q-Learning uses $\max_a Q(s_{t+1},a)$?
- Why must the training loop reuse `next_action`?
- Why does `terminated` remove bootstrapping while `truncated` does not?

In [ ]:
returns = np.asarray(episode_returns)
window = min(100, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(np.arange(window - 1, len(returns)), moving_average, label=f"{window}-episode average")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("SARSA training on Taxi-v4")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 5. Evaluate without exploration

Training returns include exploratory actions. Evaluate the learned table with the greedy policy to measure what it has learned.

In [ ]:
env = gym.make("Taxi-v4", render_mode="human")
episode_returns = []

for episode in range(5):
    state, _ = env.reset()
    episode_return = 0.0
    for step in range(200):
        action = greedy_action(state)
        state, reward, terminated, truncated, _ = env.step(action)
        episode_return += reward
        if terminated or truncated:
            break
    episode_returns.append(episode_return)
    print(f"Episode {episode + 1}: steps = {step + 1}, return = {episode_return}")

env.close()
print(f"Mean return: {np.mean(episode_returns):.1f} +/- {np.std(episode_returns):.1f}")